## Loading Dataset from Catlog and Create DataFrsmes

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()
# DBTITLE 1,Read in the data
# Read in the data
df = spark.read.csv("/Volumes/john/schema/vol/C1.csv", header=True)
df.show()

+----------+-------+--------+----------+-----+--------+
|CustomerID|   Name| Country|  JoinDate|Sales|Category|
+----------+-------+--------+----------+-----+--------+
|       101|  Alice|     USA|15-01-2023|  250|       A|
|       102|    Bob|   india|01-05-2023|  150|       B|
|       103|Charlie|      UK|20-02-2023|blank|       C|
|       104|  Alice|     USA|15-01-2023|  250|       A|
|       105|    Eve|      UK|01-03-2023|  300|   Blank|
|       106|Mallory|New York|03-15-2023|  400|       B|
|       107|  Trent|   india|10-04-2023|  350|       B|
|       108|    Bob|   India|05-01-2023|  150|       B|
|       109|  Oscar|     USA|28-02-2023|  500|       A|
|       110|  Peggy|      UK|     blank|  450|       C|
+----------+-------+--------+----------+-----+--------+



Handles Blank values to Null


### Handling Blanks

In [0]:
df=df.replace(["blank","Blank"],None)
display(df)

CustomerID,Name,Country,JoinDate,Sales,Category
101,Alice,USA,15-01-2023,250,A
102,Bob,india,01-05-2023,150,B
103,Charlie,UK,20-02-2023,null,C
104,Alice,USA,15-01-2023,250,A
105,Eve,UK,01-03-2023,300,null
106,Mallory,New York,03-15-2023,400,B
107,Trent,india,10-04-2023,350,B
108,Bob,India,05-01-2023,150,B
109,Oscar,USA,28-02-2023,500,A
110,Peggy,UK,null,450,C


### To Transform Country name (india to India)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as F
df=df.withColumn(
    "Country",
    when(F.col("Country")==F.upper(F.col("Country")),F.col("Country")).otherwise(initcap(F.col("Country"))))
display(df)

CustomerID,Name,Country,JoinDate,Sales,Category
101,Alice,USA,15-01-2023,250,A
102,Bob,India,01-05-2023,150,B
103,Charlie,UK,20-02-2023,null,C
104,Alice,USA,15-01-2023,250,A
105,Eve,UK,01-03-2023,300,null
106,Mallory,New York,03-15-2023,400,B
107,Trent,India,10-04-2023,350,B
108,Bob,India,05-01-2023,150,B
109,Oscar,USA,28-02-2023,500,A
110,Peggy,UK,null,450,C


### To Replace State to Country

In [0]:
df=df.replace("New York","USA")
display(df)

CustomerID,Name,Country,JoinDate,Sales,Category
101,Alice,USA,15-01-2023,250,A
102,Bob,India,01-05-2023,150,B
103,Charlie,UK,20-02-2023,null,C
104,Alice,USA,15-01-2023,250,A
105,Eve,UK,01-03-2023,300,null
106,Mallory,USA,03-15-2023,400,B
107,Trent,India,10-04-2023,350,B
108,Bob,India,05-01-2023,150,B
109,Oscar,USA,28-02-2023,500,A
110,Peggy,UK,null,450,C


### To remove Duplicates

In [0]:
df = df.dropDuplicates()
display(df)

CustomerID,Name,Country,JoinDate,Sales,Category
105,Eve,UK,01-03-2023,300,null
109,Oscar,USA,28-02-2023,500,A
103,Charlie,UK,20-02-2023,null,C
108,Bob,India,05-01-2023,150,B
106,Mallory,USA,03-15-2023,400,B
107,Trent,India,10-04-2023,350,B
104,Alice,USA,15-01-2023,250,A
101,Alice,USA,15-01-2023,250,A
102,Bob,India,01-05-2023,150,B
110,Peggy,UK,null,450,C


In [0]:
df = df.withColumn("Sales",
    when(col("Sales").cast("int").isNotNull(), col("Sales").cast("int"))
    .otherwise(None)
)
display(df)

CustomerID,Name,Country,JoinDate,Sales,Category
105,Eve,UK,01-03-2023,300,null
109,Oscar,USA,28-02-2023,500,A
103,Charlie,UK,20-02-2023,null,C
108,Bob,India,05-01-2023,150,B
106,Mallory,USA,03-15-2023,400,B
107,Trent,India,10-04-2023,350,B
104,Alice,USA,15-01-2023,250,A
101,Alice,USA,15-01-2023,250,A
102,Bob,India,01-05-2023,150,B
110,Peggy,UK,null,450,C


### To Fill Null Values

In [0]:
df = df.fillna({
    "Sales": 0,
    "Category": "UNKNOWN"
})
display(df)

CustomerID,Name,Country,JoinDate,Sales,Category
105,Eve,UK,01-03-2023,300,UNKNOWN
109,Oscar,USA,28-02-2023,500,A
103,Charlie,UK,20-02-2023,0,C
108,Bob,India,05-01-2023,150,B
106,Mallory,USA,03-15-2023,400,B
107,Trent,India,10-04-2023,350,B
104,Alice,USA,15-01-2023,250,A
101,Alice,USA,15-01-2023,250,A
102,Bob,India,01-05-2023,150,B
110,Peggy,UK,null,450,C


### Removing Duplicate

In [0]:
df = df.filter(col("CustomerID") != 104)
df = df.orderBy("CustomerID")
display(df)

CustomerID,Name,Country,JoinDate,Sales,Category
101,Alice,USA,15-01-2023,250,A
102,Bob,India,01-05-2023,150,B
103,Charlie,UK,20-02-2023,0,C
105,Eve,UK,01-03-2023,300,UNKNOWN
106,Mallory,USA,03-15-2023,400,B
107,Trent,India,10-04-2023,350,B
108,Bob,India,05-01-2023,150,B
109,Oscar,USA,28-02-2023,500,A
110,Peggy,UK,null,450,C
